# Genome browser with IGV
Opened from **BNERC**, the research database of the Bio-Nitrogen Economy Research Center (BicBioEng Lab, University of South Dakota).

Downloads each selected genome (and, for NCBI assemblies, its gene annotation) and opens it in an interactive IGV genome browser ([igv-notebook](https://github.com/igvteam/igv-notebook)). Zoom, search a gene name or locus, and click a gene for details.

**How to run:** paste the records BNERC copied into `RECORDS` below, then choose **Runtime → Run all**. The first 3 genomes are shown.

In [ ]:
#@title Records to analyze
#@markdown Paste what BNERC copied for you: `accession:source` pairs separated by commas. The source can be left out; it's inferred from the accession.
RECORDS = "GCA_000009705:cyanobacteria_ena, GCA_000020025:cyanobacteria_ena"  #@param {type:"string"}
#@markdown NCBI asks for a contact email with each download request.
NCBI_EMAIL = "readstech@gmail.com"  #@param {type:"string"}

def _parse(text):
    out = []
    for part in text.replace("\n", ",").split(","):
        part = part.strip()
        if not part:
            continue
        acc, _, src = part.partition(":")
        acc, src = acc.strip(), src.strip()
        if not src:
            src = "cyanobacteria_ncbi" if acc.upper().startswith(("GCA_", "GCF_")) else "cyanorak"
        out.append((acc, src))
    return out

records = _parse(RECORDS)
accessions = [a for a, _ in records]
assert records, "Paste at least one record into RECORDS above."
print(f"{len(records)} record(s):", ", ".join(f"{a} ({s})" for a, s in records))

## 1. Install igv-notebook

In [ ]:
%pip install -q igv-notebook biopython
DATASETS_INCLUDE = "genome,gff3"  # NCBI assemblies: sequence plus gene annotation

## 2. Download sequences and annotation

In [ ]:
#@title Download genomes as FASTA (.fna)
import glob, os, shutil, subprocess, sys, urllib.request, zipfile
try:
    from Bio import Entrez
except ImportError:  # the install cell was skipped, or its pip went to another Python
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "biopython"], check=True)
    from Bio import Entrez
Entrez.email = NCBI_EMAIL

def _datasets_cli():
    exe = shutil.which("datasets") or os.path.abspath("bin/datasets")
    if not os.path.exists(exe):
        os.makedirs("bin", exist_ok=True)
        urllib.request.urlretrieve("https://ftp.ncbi.nlm.nih.gov/pub/datasets/command-line/LATEST/linux-amd64/datasets", exe)
        os.chmod(exe, 0o755)
    return exe

def fetch(acc, source):
    dest = f"{acc}.fna"
    if os.path.exists(dest):
        return print(f"✅ {dest} already downloaded")
    try:
        if source in ("cyanobacteria_ncbi", "cyanobacteria_gtdb"):
            # Assembly accessions (GCA_/GCF_): NCBI Datasets CLI
            subprocess.run([_datasets_cli(), "download", "genome", "accession", acc, "--include", globals().get("DATASETS_INCLUDE", "genome"), "--filename", f"{acc}.zip"], check=True)
            with zipfile.ZipFile(f"{acc}.zip") as z:
                z.extractall(f"genome_{acc}")
            hits = glob.glob(f"genome_{acc}/ncbi_dataset/data/*/*.fna")
            if not hits:
                return print(f"❌ No .fna in the NCBI package for {acc}")
            shutil.move(hits[0], dest)
        elif source == "cyanobacteria_ena":
            urllib.request.urlretrieve(f"https://www.ebi.ac.uk/ena/browser/api/fasta/{acc}?download=true", dest)
        else:
            # cyanorak / cyanobacteria_ncbi_nucelotide: NCBI Nucleotide via Entrez
            with Entrez.efetch(db="nucleotide", id=acc, rettype="fasta", retmode="text") as h:
                open(dest, "w").write(h.read())
        print(f"✅ {dest} ({os.path.getsize(dest) // 1024} KB)")
    except Exception as e:
        print(f"❌ {acc} ({source}): {e}")

for acc, src in records:
    fetch(acc, src)
missing = [a for a in accessions if not os.path.exists(f"{a}.fna") or os.path.getsize(f"{a}.fna") == 0]
if missing:
    print("⚠️ Not downloaded:", ", ".join(missing))

## 3. Browse

In [ ]:
import igv_notebook
igv_notebook.init()

def faidx(path):
    # Writes the samtools-style .fai index igv.js needs (one fixed line width per sequence).
    rows, name, pos = [], None, 0
    with open(path, "rb") as f:
        for line in f:
            if line.startswith(b">"):
                if name:
                    rows.append((name, length, start, bases, width))
                name, length, start, bases, width = line[1:].split()[0].decode(), 0, pos + len(line), 0, 0
            elif name and line.strip():
                n = len(line.rstrip(b"\r\n"))
                if not bases:
                    bases, width = n, len(line)
                length += n
            pos += len(line)
    if name:
        rows.append((name, length, start, bases, width))
    with open(path + ".fai", "w") as out:
        out.writelines("\t".join(map(str, r)) + "\n" for r in rows)
    return rows

for acc in [a for a in accessions if os.path.exists(f"{a}.fna") and os.path.getsize(f"{a}.fna")][:3]:
    seqs = faidx(f"{acc}.fna")
    gff = glob.glob(f"genome_{acc}/ncbi_dataset/data/*/genomic.gff")
    tracks = [{"name": "Genes", "path": gff[0], "format": "gff3", "type": "annotation", "displayMode": "EXPANDED", "filterTypes": ["chromosome", "gene", "region"]}] if gff else []
    print(f"{acc}: {len(seqs)} sequence(s), {sum(r[1] for r in seqs):,} bp" + ("" if gff else " (no gene annotation available)"))
    igv_notebook.Browser({"reference": {"id": acc, "name": acc, "fastaPath": f"{acc}.fna", "indexPath": f"{acc}.fna.fai"}, "locus": seqs[0][0], "tracks": tracks})